# ConDetection-DANN — Publication Figures

Thin orchestrator notebook. All model/training code lives in `src/`.
Run `scripts/train.py` first to produce checkpoint + training history.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import torch

from src.config import load_config
from src.data.datasets import build_splits, make_loaders
from src.models.factory import get_model
from src.training.losses import build_criterion
from src.training.trainer import evaluate
from src.visualization.figures import (
    plot_ablation_chart,
    plot_confusion_matrices,
    plot_cross_scale_attention_heatmap,
    plot_generalization_gap,
    plot_roc_curves,
    plot_score_distributions,
    plot_sota_comparison,
    plot_training_curves,
)

cfg = load_config("../configs/default.yaml")
cfg.make_dirs()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FIG_DIR = cfg.paths.figures_dir
print(f"Device: {device}  |  Figures -> {FIG_DIR}")

## 1. Training curves

In [ ]:
history_path = Path(cfg.paths.output_dir) / "training_history.csv"
if history_path.exists():
    history = pd.read_csv(history_path).to_dict("records")
    plot_training_curves(history, f"{FIG_DIR}/training_curves.png")
    print(f"Saved training_curves.png ({len(history)} epochs)")
else:
    print(f"Not found: {history_path} — run scripts/train.py first")

## 2. Model evaluation figures

In [ ]:
CKPT = Path(cfg.paths.checkpoint_dir) / "model_best.pt"
assert CKPT.exists(), f"Checkpoint not found: {CKPT}"

model = get_model("condetection", cfg).to(device)
model.load_state_dict(torch.load(str(CKPT), map_location=device, weights_only=True))
model.eval()

_, _, for_test_df, itw_df = build_splits(cfg)
_, _, for_test_loader, itw_loader = make_loaders(
    pd.DataFrame(), pd.DataFrame(), for_test_df, itw_df, cfg
)

thr_file = Path(cfg.paths.checkpoint_dir) / "best_threshold.txt"
threshold = float(thr_file.read_text().strip()) if thr_file.exists() else 0.5
print(f"Threshold: {threshold:.4f}")

criterion = build_criterion(cfg, device)
for_m = evaluate(model, for_test_loader, criterion, device, cfg, threshold=threshold)
itw_m = evaluate(model, itw_loader, criterion, device, cfg, threshold=threshold)

print(f"FoR Test | EER={for_m['EER']:.4f} | AUC={for_m['AUC']:.4f} | F1={for_m['F1']:.4f}")
print(f"ITW      | EER={itw_m['EER']:.4f} | AUC={itw_m['AUC']:.4f} | F1={itw_m['F1']:.4f}")

In [ ]:
results = {"FoR Test": for_m, "In-the-Wild": itw_m}
plot_roc_curves(results, f"{FIG_DIR}/roc_curves.png")
plot_confusion_matrices(results, f"{FIG_DIR}/confusion_matrices.png", threshold=threshold)
plot_score_distributions(results, f"{FIG_DIR}/score_distributions.png")

# Cross-scale attention heatmap
m_inner = model.module if hasattr(model, "module") else model
attn_w = getattr(m_inner, "_last_attn_weights", None)
if attn_w is not None:
    plot_cross_scale_attention_heatmap(
        attn_w,
        [mc.name for mc in cfg.mel_configs],
        f"{FIG_DIR}/attention_heatmap.png",
    )
    print("Saved: ROC, confusion, score distributions, attention heatmap")
else:
    print("Saved: ROC, confusion, score distributions")

## 3. SOTA comparison

In [ ]:
comp_path = Path(cfg.paths.tables_dir) / "comparative_results.csv"
if comp_path.exists():
    df_comp = pd.read_csv(comp_path)
    plot_sota_comparison(df_comp, f"{FIG_DIR}/sota_comparison.png")
    gap_data = {row["model"]: (row["for_eer"], row["itw_eer"]) for _, row in df_comp.iterrows()}
    plot_generalization_gap(gap_data, f"{FIG_DIR}/generalization_gap.png")
    display(df_comp[["model", "params_M", "for_eer", "for_auc", "itw_eer", "itw_auc", "gen_gap_eer"]])
else:
    print(f"Not found: {comp_path} — run scripts/run_comparison.py first")

## 4. Ablation study

In [ ]:
abl_path = Path(cfg.paths.tables_dir) / "ablation_results.csv"
if abl_path.exists():
    df_abl = pd.read_csv(abl_path)
    plot_ablation_chart(df_abl, f"{FIG_DIR}/ablation_chart.png")
    display(df_abl[["name", "description", "for_eer", "itw_eer", "gen_gap_eer"]])
else:
    print(f"Not found: {abl_path} — run scripts/run_ablation.py first")

## 5. t-SNE domain separation (optional — slow)

In [ ]:
# Uncomment to generate t-SNE (takes 5-10 min)
# from src.visualization.tsne import plot_tsne_domains
# cfg_no_dann = load_config("../configs/default.yaml")
# cfg_no_dann.dann.enabled = False
# model_no_dann = get_model("condetection", cfg_no_dann).to(device)
# plot_tsne_domains(
#     model, model_no_dann,
#     for_test_loader, itw_loader,
#     device, cfg.mel_configs, cfg.audio.sample_rate,
#     f"{FIG_DIR}/tsne_domain_separation.png",
# )
# print("Saved tsne_domain_separation.png")
print("t-SNE commented out. Uncomment cell to generate.")